# 📊 Análisis Bibliométrico — TEA & Altas Capacidades en Academia
### *Corpus completo con datos de citas, keywords, abstracts, países y acceso abierto*
**Fuente:** Lens.org · CSV export (campos extendidos) + BibTeX  
**Versión:** 2.0 — Junio 2026  
**Repositorio:** github.com/nsiico/tea-aacc-bibliometria

---
| Sección | Contenido |
|---|---|
| §1 | Setup y carga de datos |
| §2 | Deduplicación y corpus maestro |
| §3 | Estadísticas generales |
| §4 | Evolución temporal + CAGR |
| §5 | Análisis de citas (h-index, top cited) |
| §6 | Revistas — Ley de Bradford |
| §7 | Autores — Ley de Lotka |
| §8 | Países y distribución geográfica |
| §9 | Acceso abierto |
| §10 | Keywords y campos de estudio |
| §11 | Análisis de abstracts — corpus relevante |
| §12 | WordCloud y términos emergentes |
| §13 | Red de co-autoría |
| §14 | Literatura iberoamericana y colombiana |
| §15 | Protocolo de autoexploración TEA — análisis instrumental |
| §16 | Resumen ejecutivo y exportación |


## §1 — Setup e importación de librerías

In [ ]:
# Instalar si es necesario:
# !pip install bibtexparser pandas matplotlib seaborn wordcloud networkx

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
from wordcloud import WordCloud
from collections import Counter
from itertools import combinations
import re, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'#FAFAF7','axes.facecolor':'#FAFAF7',
    'axes.spines.top':False,'axes.spines.right':False,
    'font.family':'DejaVu Sans','axes.titlesize':13,'axes.labelsize':11,
})
PALETTE = ['#2e4a7a','#c0392b','#5c7a2e','#d4820a','#7a2e6b','#0a7a6b']
print('✅ Librerías cargadas')


## §2 — Carga de datos y deduplicación

Los CSV de Lens.org contienen campos extendidos ausentes en el BibTeX: **citas recibidas, abstract, keywords, campos de estudio, país, acceso abierto, financiamiento y referencias**.


In [ ]:
# ── Rutas — ajustar si es necesario ──
CSV_FILES = {
    'S1 — Amplio':                'string-1.csv',
    'S2 — Adultos Profesionales': 'string-2.csv',
    'S3 — Diagnóstico Tardío':    'string-3.csv',
    'S4 — Neurodiversidad Laboral':'string-4.csv',
    'S5 — Español/Latinoamérica': 'string-5.csv',
}

raw = {}
for label, path in CSV_FILES.items():
    df = pd.read_csv(path, low_memory=False)
    df['string'] = label
    raw[label] = df
    print(f'  {label:35s} → {len(df):4d} registros')

# Corpus maestro — deduplicar por Lens ID
master = pd.concat(raw.values(), ignore_index=True)
master = master.drop_duplicates(subset='Lens ID', keep='first').copy()

# Campos derivados
master['year_num']  = pd.to_numeric(master['Publication Year'], errors='coerce')
master['citations'] = pd.to_numeric(master['Citing Works Count'], errors='coerce').fillna(0)
master['is_article']= master['Publication Type'].str.lower().str.contains('journal article', na=False)

print(f'\nCorpus maestro: {len(master)} registros únicos')
print(f'Duplicados eliminados: {sum(len(v) for v in raw.values()) - len(master)}')


In [ ]:
# ── Sub-corpus relevante: TEA + AACC (título O abstract) ──
def is_relevant(row):
    txt = (str(row.get('Title','')) + ' ' + str(row.get('Abstract',''))).lower()
    asd  = any(x in txt for x in ['autism','asd','autistic','asperger','espectro autista'])
    gift = any(x in txt for x in ['gifted','giftedness','twice-except','twice exceptional',
                                   'high abilit','altas capacidades','superdot','doble excep',
                                   'dual except','intellectual gift'])
    return asd and gift

rel = master[master.apply(is_relevant, axis=1)].copy()
print(f'Sub-corpus TEA+AACC (título/abstract): {len(rel)} registros ({100*len(rel)/len(master):.1f}%)')


## §3 — Estadísticas generales del corpus

In [ ]:
summary = []
for label, df in raw.items():
    df2 = df.copy()
    df2['cit'] = pd.to_numeric(df2['Citing Works Count'], errors='coerce').fillna(0)
    years = pd.to_numeric(df2['Publication Year'], errors='coerce').dropna()
    summary.append({
        'String':       label.split(' — ')[0],
        'N':            len(df2),
        'Artículos':    (df2['Publication Type'].str.lower().str.contains('journal article', na=False)).sum(),
        'OA (%)':       f"{100*df2['Is Open Access'].astype(str).str.lower().str.contains('true').mean():.0f}%",
        'Año min':      int(years.min()) if len(years) else '-',
        'Año max':      int(years.max()) if len(years) else '-',
        'Citas totales':int(df2['cit'].sum()),
        'Media citas':  f"{df2['cit'].mean():.1f}",
    })
df_sum = pd.DataFrame(summary)
print(df_sum.to_string(index=False))

# h-index per sub-corpus
print('\nh-index estimado por sub-corpus:')
for label, df in raw.items():
    cits = sorted(pd.to_numeric(df['Citing Works Count'],errors='coerce').fillna(0).tolist(), reverse=True)
    h = sum(1 for i, c in enumerate(cits) if c >= i+1)
    print(f'  {label.split(" — ")[0]:10s}: h={h}')

# Corpus master h-index
all_cits = sorted(master['citations'].tolist(), reverse=True)
h_master = sum(1 for i,c in enumerate(all_cits) if c >= i+1)
h_rel    = sum(1 for i,c in enumerate(sorted(rel['citations'].tolist(),reverse=True)) if c >= i+1)
print(f'\n  Corpus maestro (dedup): h={h_master}')
print(f'  Sub-corpus TEA+AACC:    h={h_rel}')


## §4 — Evolución temporal y CAGR

In [ ]:
df_yr = master[(master['year_num']>=1990)&(master['year_num']<=2026)].copy()
yearly = df_yr.groupby('year_num').size().reset_index(name='n')

def cagr(series, y0, y1):
    s = series.set_index('year_num')['n']
    if y0 in s.index and y1 in s.index and s[y0]>0:
        return ((s[y1]/s[y0])**(1/(y1-y0))-1)*100
    return None

print('CAGR del corpus maestro:')
for p in [(2000,2010),(2010,2020),(2015,2022),(2020,2025)]:
    c = cagr(yearly, *p)
    if c: print(f'  {p[0]}–{p[1]}: {c:+.1f}%/año')

post2020 = (df_yr['year_num']>=2020).sum()
total    = len(df_yr)
print(f'\nPost-2020: {post2020}/{total} = {100*post2020/total:.1f}%')
print(f'Post-2015: {(df_yr["year_num"]>=2015).sum()}/{total} = '
      f'{100*(df_yr["year_num"]>=2015).sum()/total:.1f}%')

# Plot
fig, axes = plt.subplots(2,1,figsize=(13,10))
ax = axes[0]
ax.bar(yearly['year_num'], yearly['n'], color='#2e4a7a', alpha=0.75, width=0.8)
mm = yearly.set_index('year_num')['n'].rolling(3,center=True).mean()
ax.plot(mm.index, mm.values, color='#c0392b', lw=2.5, label='Media móvil 3a')
ax.axvline(2013, color='gray', ls='--', lw=1, alpha=0.6, label='DSM-5 (2013)')
ax.axvline(2020, color='#5c7a2e', ls='--', lw=1, alpha=0.6, label='COVID-19 (2020)')
ax.set_title('Evolución temporal — Corpus maestro (TEA+AACC+Academia)', fontweight='bold')
ax.set_xlim(1995,2027); ax.legend(fontsize=9)

ax2 = axes[1]
for i,(label,df) in enumerate(raw.items()):
    s = pd.to_numeric(df['Publication Year'],errors='coerce')
    s = s[(s>=2005)&(s<=2026)].value_counts().sort_index()
    ax2.plot(s.index, s.values, marker='o', ms=4, lw=1.8,
             color=PALETTE[i], label=label.split(' — ')[0])
ax2.set_title('Evolución por string (2005–2026)', fontweight='bold')
ax2.legend(fontsize=9); ax2.set_xlim(2005,2027)
plt.tight_layout()
plt.savefig('fig_02_temporal_v2.png', dpi=150, bbox_inches='tight')
plt.show()


## §5 — Análisis de citas: h-index, distribución y top citados

In [ ]:
# Distribución de citas
fig, axes = plt.subplots(1,2,figsize=(13,5))

# Histograma (excl outliers)
cit_filtered = master[master['citations']<200]['citations']
axes[0].hist(cit_filtered, bins=40, color=PALETTE[0], alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Citas recibidas')
axes[0].set_ylabel('N documentos')
axes[0].set_title('Distribución de citas (excl. >200)', fontweight='bold')
axes[0].axvline(master['citations'].median(), color=PALETTE[1], lw=2,
                 label=f'Mediana={master["citations"].median():.0f}')
axes[0].axvline(master['citations'].mean(), color=PALETTE[2], lw=2, ls='--',
                 label=f'Media={master["citations"].mean():.1f}')
axes[0].legend(fontsize=9)

# h-index visual (Hirsch plot)
cits_sorted = sorted(master['citations'].tolist(), reverse=True)[:200]
ranks = range(1, len(cits_sorted)+1)
axes[1].plot(list(ranks), cits_sorted, color=PALETTE[0], lw=2)
axes[1].plot(list(ranks), list(ranks), color='gray', ls='--', lw=1, label='y=x')
h = sum(1 for i,c in enumerate(cits_sorted) if c >= i+1)
axes[1].axvline(h, color=PALETTE[1], lw=2, label=f'h-index = {h}')
axes[1].axhline(h, color=PALETTE[1], lw=2)
axes[1].set_xlabel('Rango'); axes[1].set_ylabel('Citas')
axes[1].set_title('Gráfico de Hirsch — h-index del corpus', fontweight='bold')
axes[1].legend(); axes[1].set_xlim(0,100); axes[1].set_ylim(0,400)
plt.tight_layout()
plt.savefig('fig_05_citations_hindex.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'h-index corpus completo: {h}')
print(f'Total citas: {int(master["citations"].sum())}')
print(f'Media: {master["citations"].mean():.2f}  Mediana: {master["citations"].median():.0f}')


In [ ]:
# Top 20 más citados — relevantes
print('TOP 20 DOCUMENTOS MÁS CITADOS (sub-corpus TEA+AACC):')
top_rel = rel.sort_values('citations',ascending=False).head(20)
for i,(_, r) in enumerate(top_rel.iterrows(),1):
    print(f'{i:2d}. [{int(r["citations"]):4d} cit] ({r["Publication Year"]}) '
          f'{str(r["Title"])[:65]}')
    print(f'    {str(r["Author/s"])[:60]} | {str(r["Source Title"])[:40]}')


## §6 — Revistas: Ley de Bradford

In [ ]:
arts = master[master['is_article'] & master['Source Title'].notna()].copy()
jcounts = arts['Source Title'].str.strip().str.lower().value_counts()
total_arts = len(arts)

# Bradford zones
cum, z1, z2, z3 = 0, [], [], []
for j, n in jcounts.items():
    cum += n
    pct = 100*cum/total_arts
    (z1 if pct<=33 else z2 if pct<=66 else z3).append((j,n))

print(f'Artículos totales: {total_arts}')
print(f'Revistas únicas:   {len(jcounts)}')
print(f'Zona 1 (núcleo):  {len(z1)} revistas')
print(f'Zona 2:           {len(z2)} revistas')
print(f'Zona 3 (periferia):{len(z3)} revistas')
print(f'\nNúcleo Bradford (top 10):')
for j,n in z1[:10]: print(f'  {n:3d}  {j[:60]}')

# Plot
fig, axes = plt.subplots(1,2,figsize=(15,6))
top20 = jcounts.head(20)
j_labels = [j[:40]+'…' if len(j)>40 else j for j in top20.index]
zone_colors = [PALETTE[0] if j in [x[0] for x in z1] else
               PALETTE[1] if j in [x[0] for x in z2] else '#aaa'
               for j in top20.index]
axes[0].barh(range(len(top20)), top20.values, color=zone_colors, height=0.7)
axes[0].set_yticks(range(len(top20))); axes[0].set_yticklabels(j_labels, fontsize=8)
axes[0].invert_yaxis(); axes[0].set_xlabel('N artículos')
axes[0].set_title('Top 20 revistas', fontweight='bold')
p1=mpatches.Patch(color=PALETTE[0],label='Zona 1 — Núcleo')
p2=mpatches.Patch(color=PALETTE[1],label='Zona 2')
p3=mpatches.Patch(color='#aaa',label='Zona 3 — Periferia')
axes[0].legend(handles=[p1,p2,p3], fontsize=8)

df_brad = pd.DataFrame({'j':jcounts.index,'n':jcounts.values})
df_brad['cum_pct'] = 100*df_brad['n'].cumsum()/df_brad['n'].sum()
df_brad['rank']    = range(1,len(df_brad)+1)
axes[1].semilogx(df_brad['rank'], df_brad['cum_pct'], color=PALETTE[0], lw=2)
axes[1].axhline(33,color=PALETTE[1],ls='--',alpha=0.7,label='33%')
axes[1].axhline(66,color=PALETTE[2],ls='--',alpha=0.7,label='66%')
axes[1].set_xlabel('Rango (log)'); axes[1].set_ylabel('% acumulado')
axes[1].set_title('Curva Bradford acumulada', fontweight='bold')
axes[1].legend(); axes[1].set_ylim(0,105)
plt.tight_layout()
plt.savefig('fig_03_bradford_v2.png', dpi=150, bbox_inches='tight')
plt.show()


## §7 — Autoría: Ley de Lotka

In [ ]:
def parse_authors(s):
    if not s or pd.isna(s): return []
    return [a.strip() for a in re.split(r';|\s+and\s+', str(s), flags=re.I) if a.strip()]

all_authors = []
for _, row in master.iterrows():
    all_authors.extend(parse_authors(row['Author/s']))
ac = Counter(all_authors)

print(f'Autores únicos: {len(ac)}')
prod = Counter(ac.values())
single = prod.get(1,0)
print(f'Con 1 pub: {single} ({100*single/len(ac):.1f}%)')
print(f'Con ≥5 pubs: {sum(v for k,v in prod.items() if k>=5)}')
print(f'Con ≥10 pubs: {sum(v for k,v in prod.items() if k>=10)}')
print('\nTop 20 autores:')
for a,n in ac.most_common(20): print(f'  {n:3d}  {a}')

fig, axes = plt.subplots(1,2,figsize=(13,5))
top20a = pd.DataFrame(ac.most_common(20), columns=['author','n'])
axes[0].barh(range(len(top20a)), top20a['n'], color=PALETTE[1], height=0.7)
axes[0].set_yticks(range(len(top20a)))
axes[0].set_yticklabels([a[:35]+'…' if len(a)>35 else a
                          for a in top20a['author']], fontsize=9)
axes[0].invert_yaxis(); axes[0].set_title('Top 20 autores más productivos', fontweight='bold')

plot_prod = pd.DataFrame(list(prod.items()),columns=['n_pubs','n_auth'])
plot_prod = plot_prod[plot_prod['n_pubs']<=20].sort_values('n_pubs')
axes[1].loglog(plot_prod['n_pubs'], plot_prod['n_auth'], 'o-', color=PALETTE[0], lw=2, ms=6)
axes[1].set_xlabel('N pubs por autor (log)'); axes[1].set_ylabel('N autores (log)')
axes[1].set_title('Ley de Lotka — Distribución de productividad', fontweight='bold')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig_04_lotka_v2.png', dpi=150, bbox_inches='tight')
plt.show()


## §8 — Distribución geográfica

In [ ]:
countries = master['Source Country'].dropna().str.strip().value_counts()
print(f'Países con publicaciones: {len(countries)}')
print('Top 20:')
print(countries.head(20).to_string())

fig, ax = plt.subplots(figsize=(10,6))
top_c = countries.head(20)
colors_c = [PALETTE[3] if 'Colombia' in c or 'Brazil' in c or 'Argentina' in c
             or 'Spain' in c or 'Mexico' in c else PALETTE[0]
             for c in top_c.index]
ax.barh(range(len(top_c)), top_c.values, color=colors_c, height=0.7)
ax.set_yticks(range(len(top_c)))
ax.set_yticklabels(top_c.index, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('N publicaciones')
ax.set_title('Distribución geográfica — Top 20 países productores', fontweight='bold')
p1=mpatches.Patch(color=PALETTE[3],label='Contexto iberoamericano')
p2=mpatches.Patch(color=PALETTE[0],label='Otros países')
ax.legend(handles=[p1,p2], fontsize=9)
plt.tight_layout()
plt.savefig('fig_11_countries.png', dpi=150, bbox_inches='tight')
plt.show()


## §9 — Acceso abierto

In [ ]:
oa_total = (master['Is Open Access'].astype(str).str.lower()=='true').sum()
total = len(master)
print(f'Open Access: {oa_total}/{total} = {100*oa_total/total:.1f}%')
print('\nPor tipo de OA:')
print(master['Open Access Colour'].value_counts().to_string())

# Tendencia OA por año
df_oa = master[(master['year_num']>=2010)&(master['year_num']<=2026)].copy()
df_oa['is_oa'] = df_oa['Is Open Access'].astype(str).str.lower()=='true'
oa_yr = df_oa.groupby('year_num')['is_oa'].agg(['sum','count'])
oa_yr['pct'] = 100*oa_yr['sum']/oa_yr['count']

fig, ax = plt.subplots(figsize=(11,4))
ax.bar(oa_yr.index, oa_yr['pct'], color=PALETTE[2], alpha=0.8, width=0.8)
ax.set_xlabel('Año'); ax.set_ylabel('% Open Access')
ax.set_title('Evolución del acceso abierto por año (2010–2026)', fontweight='bold')
ax.set_ylim(0,105)
plt.tight_layout()
plt.savefig('fig_12_openaccess.png', dpi=150, bbox_inches='tight')
plt.show()


## §10 — Keywords y campos de estudio

In [ ]:
# Keywords
all_kw = []
for kw in master['Keywords'].dropna():
    all_kw.extend([k.strip().lower() for k in str(kw).split(';') if k.strip()])
kw_freq = Counter(all_kw)
print('Top 30 keywords:')
for kw, n in kw_freq.most_common(30): print(f'  {n:4d}  {kw}')

# Keywords en relevantes
rel_kw = []
for kw in rel['Keywords'].dropna():
    rel_kw.extend([k.strip().lower() for k in str(kw).split(';') if k.strip()])
rel_kw_freq = Counter(rel_kw)
print('\nTop 20 keywords en sub-corpus TEA+AACC:')
for kw, n in rel_kw_freq.most_common(20): print(f'  {n:4d}  {kw}')

# Plot keywords comparison
fig, axes = plt.subplots(1,2,figsize=(14,6))
top_kw = pd.DataFrame(kw_freq.most_common(20), columns=['kw','n'])
axes[0].barh(range(len(top_kw)), top_kw['n'], color=PALETTE[0], height=0.7)
axes[0].set_yticks(range(len(top_kw)))
axes[0].set_yticklabels(top_kw['kw'], fontsize=9)
axes[0].invert_yaxis(); axes[0].set_title('Top 20 keywords — corpus maestro', fontweight='bold')

top_rkw = pd.DataFrame(rel_kw_freq.most_common(20), columns=['kw','n'])
axes[1].barh(range(len(top_rkw)), top_rkw['n'], color=PALETTE[1], height=0.7)
axes[1].set_yticks(range(len(top_rkw)))
axes[1].set_yticklabels(top_rkw['kw'], fontsize=9)
axes[1].invert_yaxis(); axes[1].set_title('Top 20 keywords — sub-corpus TEA+AACC', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_13_keywords.png', dpi=150, bbox_inches='tight')
plt.show()


## §11 — Análisis de abstracts del sub-corpus relevante

In [ ]:
# Instrumentos mencionados en abstracts
instruments = {
    'ADOS-2':        r'ados.?2|ados.*module',
    'RAADS-R':       r'raads',
    'AQ / AQ-28':   r'autism.quotient|aq[- ]?\d+',
    'CAT-Q':         r'cat.?q|camouflaging autistic',
    'WAIS/WISC':     r'wais|wisc|wechsler',
    'ADOS':          r'\bados\b',
    'SRS':           r'\bsrs\b',
    'ADI-R':         r'adi.?r',
    'BRIEF':         r'\bbrief\b',
    'IPA (method)':  r'interpretative phenomeno',
    'Mixed methods': r'mixed.method',
    'Qualitative':   r'qualitative',
    'RCT':           r'randomized|randomised|rct',
}
print('Menciones de instrumentos/métodos en abstracts del sub-corpus TEA+AACC:')
for name, pat in instruments.items():
    n = rel['Abstract'].str.lower().str.contains(pat, na=False, regex=True).sum()
    if n > 0:
        print(f'  {name:25s}: {n:3d} documentos')

# Metodología dominante
print('\nPapers con masking/camouflage en abstract:')
mask_papers = rel[rel['Abstract'].str.lower().str.contains('mask|camouflage|camouflaging', na=False)]
for _,r in mask_papers.sort_values('citations',ascending=False).iterrows():
    print(f'  [{int(r["citations"]):3d} cit] {r["Publication Year"]} | {str(r["Title"])[:70]}')


## §12 — WordCloud y términos emergentes pre/post 2020

In [ ]:
STOP = {'the','a','an','of','in','and','to','for','with','on','by','from','is','are',
        'was','were','be','been','this','that','at','as','it','its','or','not','but',
        'their','they','we','has','have','had','using','based','study','analysis',
        'approach','review','paper','new','two','three','one','across','among',
        'within','between','also','were','when','which','these','those','been',
        'de','la','el','en','los','las','un','una','del','para','con','se','por','al',
        'es','su','y','e','o','u','como','más','pero','sus','que','do','does','its'}

def tokenize(df_col):
    text = ' '.join(df_col.dropna().astype(str))
    return [t for t in re.findall(r'[a-záéíóúñüA-Z]{3,}', text.lower()) if t not in STOP]

# Corpus maestro
tok_all   = tokenize(master['Title'])
# Pre/post 2020
pre  = master[master['year_num']<2020]
post = master[master['year_num']>=2020]
tok_pre  = tokenize(pre['Title'])
tok_post = tokenize(post['Title'])

freq_all  = Counter(tok_all)
freq_pre  = Counter(tok_pre)
freq_post = Counter(tok_post)

# Growth ratio post/pre
tot_pre, tot_post = sum(freq_pre.values()), sum(freq_post.values())
growth = {t: (freq_post.get(t,0)/tot_post) / (freq_pre.get(t,0.01)/tot_pre)
          for t in freq_post if freq_post[t]>=5}
print('Términos con mayor crecimiento relativo post-2020 (≥5 ocurrencias):')
for t, r in sorted(growth.items(), key=lambda x:-x[1])[:20]:
    print(f'  ×{r:.1f}  {t}')

# WordCloud
fig, axes = plt.subplots(1,2,figsize=(15,6))
wc = WordCloud(width=650,height=380,background_color='#FAFAF7',
               colormap='Blues',max_words=100)\
     .generate_from_frequencies(freq_all)
axes[0].imshow(wc, interpolation='bilinear'); axes[0].axis('off')
axes[0].set_title('WordCloud — Corpus maestro', fontweight='bold')

top30 = pd.DataFrame(freq_all.most_common(30), columns=['term','freq'])
axes[1].barh(range(len(top30)), top30['freq'], color=PALETTE[0], height=0.7)
axes[1].set_yticks(range(len(top30))); axes[1].set_yticklabels(top30['term'], fontsize=9)
axes[1].invert_yaxis(); axes[1].set_title('Top 30 términos en títulos', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_06_wordcloud_v2.png', dpi=150, bbox_inches='tight')
plt.show()


## §13 — Red de co-autoría

In [ ]:
G = nx.Graph()
ew = Counter()
eligible = {a for a,n in ac.items() if n>=3}
for _,row in master.iterrows():
    auths = [a for a in parse_authors(row['Author/s']) if a in eligible]
    for a,b in combinations(sorted(auths),2):
        ew[(a,b)] += 1
for auth in eligible: G.add_node(auth, pubs=ac[auth])
for (a,b),w in ew.items(): G.add_edge(a,b,weight=w)
G.remove_nodes_from(list(nx.isolates(G)))

print(f'Nodos: {G.number_of_nodes()}')
print(f'Aristas: {G.number_of_edges()}')
print(f'Componentes: {nx.number_connected_components(G)}')

degree_c = nx.degree_centrality(G)
between  = nx.betweenness_centrality(G, weight='weight')
print('\nTop 10 por betweenness:')
for a,v in sorted(between.items(),key=lambda x:-x[1])[:10]:
    print(f'  {a[:40]:40s}  btw={v:.3f}')

fig, ax = plt.subplots(figsize=(13,9))
ax.set_facecolor('#0d0d14'); fig.patch.set_facecolor('#0d0d14')
pos = nx.spring_layout(G, k=2.5, seed=42, iterations=60)
node_sz  = [G.nodes[n]['pubs']*80 for n in G.nodes()]
node_col = [between.get(n,0) for n in G.nodes()]
nx.draw_networkx_edges(G,pos,ax=ax,alpha=0.2,edge_color='#4a90d9',
                       width=[d.get('weight',1)*0.7 for _,_,d in G.edges(data=True)])
sc=nx.draw_networkx_nodes(G,pos,ax=ax,node_size=node_sz,
                          node_color=node_col,cmap=plt.cm.YlOrRd,alpha=0.9)
top15 = dict(sorted(between.items(),key=lambda x:-x[1])[:15])
nx.draw_networkx_labels(G,pos,
    labels={n:n.split(',')[0].split(' ')[-1] for n in top15},
    font_color='white',font_size=7,ax=ax)
plt.colorbar(sc,ax=ax,label='Betweenness centrality',shrink=0.5)
ax.set_title('Red de co-autoría — TEA+AACC en Academia',
             color='white',fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('fig_07_network_v2.png', dpi=150, bbox_inches='tight',
            facecolor='#0d0d14')
plt.show()


## §14 — Literatura iberoamericana y contexto colombiano

In [ ]:
s5 = master[master['string']=='S5 — Español/Latinoamérica'].copy()
print(f'S5 (español/LATAM): {len(s5)} registros')

# Colombia específico
col_mask = (
    master['Title'].str.contains('colombi|bogot|medell|sincelej|sucre', case=False, na=False) |
    master['Source Title'].str.contains('colombi', case=False, na=False) |
    master['Abstract'].str.contains('colombi', case=False, na=False)
)
col_papers = master[col_mask].sort_values('year_num', ascending=False)
print(f'\nRegistros con mención a Colombia: {len(col_papers)}')
for _,r in col_papers.iterrows():
    print(f'  {r["Publication Year"]} | {str(r["Title"])[:70]}')
    print(f'    Citas: {int(r["citations"])} | {str(r["Source Title"])[:50]}')

# LATAM countries
latam = ['Colombia','Brazil','Argentina','Mexico','Chile','Peru','Venezuela',
         'Uruguay','Ecuador','Bolivia','Paraguay','Cuba','Costa Rica','Panama']
print('\nDistribución LATAM:')
for country in latam:
    n = (master['Source Country']==country).sum()
    if n > 0: print(f'  {country}: {n}')


## §15 — Protocolo de autoexploración TEA en adultos

Esta sección analiza los instrumentos validados disponibles en el corpus y construye un protocolo
de autoexploración estructurada para uso en investigación universitaria.
**⚠️ No es un diagnóstico. Uso orientativo únicamente.**


In [ ]:
# Análisis de instrumentos en el corpus relevante
instrument_map = {
    'ADOS-2 Módulo 4':     r'ados.?2|ados.*module.?4|ados.*adult',
    'RAADS-R':             r'raads',
    'AQ / AQ-28':         r'autism.quotient|aq[- ]?28|aq[- ]?10',
    'CAT-Q':               r'cat.?q|camouflaging.autistic.trait',
    'BRIEF-A':             r'brief.?a|behavior.rating.inventory.executive',
    'WAIS-IV':             r'wais.?iv|wais.?4',
    'S&W Heuristic':       r's.*w.heuristic|strengths.*weaknesses.*heuristic',
    'Dynamic Assessment':  r'dynamic.assess',
    'IPA (qualitative)':   r'interpretative.phenomenological|ipa',
}

print('=== INSTRUMENTOS PRESENTES EN EL CORPUS ===')
print(f'{"Instrumento":<30} {"Corpus maestro":>15} {"Sub-corpus rel":>15}')
print('-'*62)
for name, pat in instrument_map.items():
    n_all = master['Abstract'].str.lower().str.contains(pat,na=False,regex=True).sum()
    n_rel = rel['Abstract'].str.lower().str.contains(pat,na=False,regex=True).sum()
    if n_all > 0:
        print(f'{name:<30} {n_all:>15} {n_rel:>15}')

print('''
=== PROTOCOLO DE AUTOEXPLORACIÓN ESTRUCTURADA ===
Basado en dominios de: ADOS-2 M4 + RAADS-R + AQ-28 + CAT-Q + Historia de vida

DOMINIO A: Comunicación social recíproca
  A1. Iniciar conversaciones espontáneas requiere esfuerzo consciente
  A2. Dificultad para gestionar turnos de habla de forma natural
  A3. Contacto visual requiere atención activa, no es espontáneo
  A4. Solo comparte emociones/experiencias cuando le preguntan directamente
  A5. Los gestos y expresiones faciales requieren 'programarse' conscientemente
  A6. Recibe feedback de hablar demasiado de sus temas de interés
  A7. Conversaciones fuera de sus intereses resultan agotadoras o vacías

DOMINIO B: Rasgos actuales y retrospectivos (RAADS-R / AQ)
  B1. Dificultades persistentes para hacer/mantener amistades
  B2. Dificultad para interpretar ironía, dobles sentidos o intenciones implícitas
  B3. Atracción intensa por temas específicos excluyendo otros por períodos prolongados
  B4. Hipersensibilidad a luz, sonido, texturas, olores o temperatura
  B5. Necesita rutinas/previsibilidad; los cambios inesperados generan malestar
  B6. Observa las interacciones sociales como 'desde afuera', como antropólogo
  B7. Dificultad para imaginar situaciones hipotéticas o juegos de fantasía

DOMINIO C: Enmascaramiento y costo social (CAT-Q)
  C1. Aprendió reglas sociales explícitamente (libros, observación, tutoriales)
  C2. Imita expresiones, tonos y posturas de otros para parecer natural
  C3. Agotamiento mental tras reuniones sociales aunque las manejó bien
  C4. Tiene scripts/respuestas ensayadas para situaciones comunes
  C5. Oculta intereses intensos o los minimiza para no parecer extraño
  C6. Ha tenido episodios de burnout/agotamiento emocional tras alta demanda social

DOMINIO D: Historia de vida temprana (entrevista clínica)
  D1. Juego diferente al de otros niños (organización, repetición, sin juego simbólico)
  D2. 'Manías', movimientos repetitivos o ritmos de calmado propios
  D3. Descrito como 'maduro intelectualmente / inmaduro socialmente'
  D4. Bullying, aislamiento o sensación persistente de 'no encajar'
  D5. Dificultades marcadas ante cambios de escuela, horario o rutina
  D6. Algún profesional mencionó 'algo social' sin evaluar formalmente

ESCALA: 0=No me identifica  1=Parcialmente/a veces  2=Claramente/siempre
INTERPRETACIÓN (orientativa, NO diagnóstica):
  Puntuación alta en A+B+C+evidencia D → Buscar evaluación clínica especializada
  Alto en C con bajo A+B → Posible compensación extrema; no descarta TEA
  Alto en B sin D → Otros diagnósticos diferenciales; historia temprana es clave
''')


## §16 — Resumen ejecutivo y exportación

In [ ]:
# Resumen completo
print('='*65)
print('  RESUMEN EJECUTIVO — CORPUS EXTENDIDO (CSV Lens.org)')
print('='*65)
print(f'Registros totales (bruto):      {sum(len(v) for v in raw.values())}')
print(f'Corpus maestro (deduplicado):   {len(master)}')
print(f'Sub-corpus TEA+AACC relevante:  {len(rel)}')
print(f'\nCITACIONES')
print(f'  Total citas (corpus maestro): {int(master["citations"].sum())}')
print(f'  Media citas/documento:        {master["citations"].mean():.2f}')
print(f'  h-index corpus completo:      {h_master}')
print(f'  h-index sub-corpus TEA+AACC:  {h_rel}')
print(f'\nACCESO ABIERTO')
oa_n = (master['Is Open Access'].astype(str).str.lower()=='true').sum()
print(f'  Open Access:  {oa_n}/{len(master)} = {100*oa_n/len(master):.1f}%')
print(f'\nGEOGRAFÍA')
top3 = master['Source Country'].value_counts().head(3)
for c, n in top3.items(): print(f'  {c}: {n}')
col_n = master['Source Country'].str.contains('Colombia',na=False).sum()
print(f'  Colombia: {col_n}')
print()

# Export enriched master
out = master[['Lens ID','Title','Author/s','Publication Year','Publication Type',
              'Source Title','Source Country','DOI','citations','Is Open Access',
              'Open Access Colour','Keywords','Fields of Study','string']].copy()
out.to_csv('corpus_maestro_enriched.csv', index=False, encoding='utf-8-sig')
print('✅ corpus_maestro_enriched.csv exportado')

# Export relevant
rel_out = rel[['Lens ID','Title','Author/s','Publication Year','Source Title',
               'DOI','citations','Abstract','Keywords','Is Open Access','string']].copy()
rel_out.to_csv('subcorpus_TEA_AACC_relevante.csv', index=False, encoding='utf-8-sig')
print('✅ subcorpus_TEA_AACC_relevante.csv exportado')
